# Setup

In [5]:
# !pip install selenium webdriver-manager pandas beautifulsoup4 lxml

In [6]:
import os
from dotenv import load_dotenv
import time
import pandas as pd
import re
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import Select
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException

from webdriver_manager.chrome import ChromeDriverManager

# Config

In [7]:
load_dotenv()

BASE_URL = "https://haims2.doh.go.th/"
USERNAME = os.getenv("USERNAME")
PASSWORD = os.getenv("PASSWORD")

TARGET_INCIDENT_ID = "1158950"

WAIT_TIME = 10

## Google Drive Config

In [8]:
from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive

gauth = GoogleAuth()
gauth.LocalWebserverAuth() 
drive = GoogleDrive(gauth)

MAIN_GDRIVE_FOLDER_ID = '126ltPgBFcu-AZjWCjErDRpLPlB2-mEKx'
print("✅ Connected Google Drive successfully!")

Your browser has been opened to visit:

    https://accounts.google.com/o/oauth2/auth?client_id=839515928127-ccb442os010d6bb2qjmh43n0s478rhg1.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8090%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fdrive&access_type=online&response_type=code

Authentication successful.
✅ Connected Google Drive successfully!


# Start Browser

In [9]:
options = Options()
options.add_argument("--start-maximized")

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
)

wait = WebDriverWait(driver, WAIT_TIME)

driver.get(BASE_URL)

# Login

In [10]:
wait.until(lambda d: d.execute_script("return document.readyState") == "complete")

username_input = wait.until(
    EC.element_to_be_clickable((By.ID, "edit-name"))
)

password_input = wait.until(
    EC.element_to_be_clickable((By.ID, "edit-pass"))
)

login_button = wait.until(
    EC.element_to_be_clickable((By.ID, "edit-submit"))
)

username_input.click()
username_input.clear()
username_input.send_keys(USERNAME)

password_input.click()
password_input.clear()
password_input.send_keys(PASSWORD)

time.sleep(5)

login_button.click()

# time.sleep(3)

wait.until(
    EC.presence_of_element_located((By.ID, "summary_main_table"))
)

print("Login Success")

Login Success


# Helper Function

In [11]:
html = driver.page_source
soup = BeautifulSoup(html,"html.parser")

### Universal Element Finder

In [12]:
def find_element(key, tag=None):

    # search by name
    el = soup.find(tag or True, {"name": key})
    if el:
        return el

    # search by id
    el = soup.find(tag or True, {"id": key})
    if el:
        return el

    # search by class
    el = soup.find(tag or True, class_=key)
    if el:
        return el

    # search label tag
    label = soup.find("label", string=lambda x: x and key in x)
    if label:
        return label.find_next()

    # search text label (span/div)
    text_label = soup.find(string=lambda x: x and key in x)
    if text_label:
        return text_label.parent

    return None

### Universal Value Extractor

In [13]:
def get_value(el):

    if not el:
        return None

    # ถ้ามี input อยู่ข้างใน
    inputs = el.find_elements(By.TAG_NAME, "input")
    if inputs:
        return inputs[0].get_attribute("value")

    # textarea
    textareas = el.find_elements(By.TAG_NAME, "textarea")
    if textareas:
        return textareas[0].get_attribute("value")

    # select
    selects = el.find_elements(By.TAG_NAME, "select")
    if selects:
        for opt in selects[0].find_elements(By.TAG_NAME, "option"):
            if opt.is_selected():
                return opt.text.strip()

    return el.text.strip()

In [14]:
def get_attr_by_id(element_id, attr):
    try:
        el = driver.find_element(By.ID, element_id)
        return el.get_attribute(attr)
    except:
        return None

### Universal Driver Element Finder

In [15]:
def get_by_key(key):
    
    try:
        el = driver.find_element(By.NAME, key)
        return el.get_attribute("value")
    except:
        pass

    try:
        el = driver.find_element(By.NAME, key)
        return get_value(el)
    except:
        pass

    try:
        el = driver.find_element(By.ID, key)
        return get_value(el)
    except:
        pass

    try:
        el = driver.find_element(By.CLASS_NAME, key)
        return get_value(el)
    except:
        pass

    return None

In [16]:
def get_by_css(css):
    el = driver.find_element(By.CSS_SELECTOR, css)
    return get_value(el)

### Checkbox & Label Finder

In [17]:
def get_checkbox_labels(name):

    results = []

    checkboxes = driver.find_elements(By.NAME, name)

    for cb in checkboxes:
        if cb.is_selected():

            cb_id = cb.get_attribute("id")

            label = driver.find_element(By.CSS_SELECTOR, f"label[for='{cb_id}']")
            results.append(label.text.strip())

    return results

# Extract

## SECTION 0: Header

In [18]:
def split_user_datetime(text):

    if not text:
        return None, None

    parts = text.rsplit(" ", 4)

    if len(parts) >= 5:
        user = parts[0]
        dt = " ".join(parts[1:])
        return user.strip(), dt.strip()

    return None, None

In [19]:
def section_0(project_id):
    userdata = driver.find_element(By.ID, "userdata").text
    lines = userdata.split("\n")

    create_text = lines[0].replace("สร้าง:", "").strip()
    update_text = lines[1].replace("แก้ไข:", "").strip()

    create_user, create_datetime = split_user_datetime(create_text)
    update_user, update_datetime = split_user_datetime(update_text)
    
    metadata = driver.find_elements(By.CSS_SELECTOR, "#metadata .d-inline-block")

    highway_reporter = get_value(metadata[0])
    receiver = get_value(metadata[1])
    sender = get_value(metadata[2])
    
    # ------ Gathering Parts ------
    
    header = {
        "project_id": project_id,
        "create_datetime": create_datetime,
        "create_user": create_user,
        "update_datetime": update_datetime,
        "update_user": update_user,
        "highway_reporter": highway_reporter,
        "receiver": receiver,
        "sender": sender,
    }

    header_df = pd.DataFrame([header])
    
    return header_df

## SECTION 1-2

In [20]:
def section_1_2(project_id):
    
    date = driver.find_element(By.ID,"date-mask").get_attribute("value")

    hour = driver.find_element(By.ID,"hour") \
        .find_element(By.CSS_SELECTOR,"option:checked").text

    minute = driver.find_element(By.ID,"minute") \
        .find_element(By.CSS_SELECTOR,"option:checked").text

    incident_datetime = f"{date} {hour}:{minute}"

    is_major_accident = 1 if driver.find_element(By.ID,"is_big_accident").is_selected() else 0

    is_public_interest = 1 if driver.find_element(By.ID,"has_public_figure").is_selected() else 0
    
    # ------ Gathering Parts ------

    section_1_2 = {
        "project_id": project_id,
        "incident_datetime": incident_datetime,
        "is_major_accident": is_major_accident,
        "is_public_interest": is_public_interest
    }

    section_1_2_df = pd.DataFrame([section_1_2])
    
    return section_1_2_df

## SECTION 3

In [21]:
def section_3(project_id):
    is_new_construction_zone = 1 if driver.find_element(By.ID,"is_new_construct_road").is_selected() else 0

    highway_district = driver.find_element(By.ID, "bureau") \
        .find_element(By.CSS_SELECTOR, "option[selected]").text
    highway_subdistrict = driver.find_element(By.ID, "district").get_attribute("value")
    highway_maintenance_unit = driver.find_element(By.ID, "depo").get_attribute("value")

    highway_number = driver.find_element(By.ID, "route").get_attribute("value")
    control_section = driver.find_element(By.ID, "controlsection").get_attribute("value")
    kilopost = driver.find_element(By.ID, "kilometre").get_attribute("value").lstrip('0') + "+" + driver.find_element(By.ID, "metre").get_attribute("value")

    roadinfo_text = driver.find_element(By.ID, "roadinfo").text
    parts = [x.strip() for x in roadinfo_text.split(" / ")]
    road_name = None
    section_name = None
    province = None
    section_start_km = None
    section_end_km = None
    if len(parts) > 0:
        km_text = parts[-1] 
        
        km = re.findall(r'\d+\+\d+', km_text)
        if len(km) >= 2:
            section_start_km = km[0]
            section_end_km = km[1]
        elif len(km) == 1:
            section_start_km = km[0]
            
        if len(parts) >= 4:
            road_name = parts[0]
            section_name = parts[1]
            province = parts[-2] 
            
        elif len(parts) == 3:
            road_name = parts[0]
            province = parts[1]
            
        elif len(parts) == 2:
            road_name = parts[0]

    road_feature = driver.find_element(By.ID, "road_char").text
    road_status = driver.find_element(By.ID, "road_condition").text
    lanes_count = driver.find_element(By.ID, "road_lane").get_attribute("value")

    road_direction = driver.find_element(By.ID, "road_direction").text
    median_type = driver.find_element(By.ID, "road_isle").text
    traffic_type = driver.find_element(By.ID, "road_traffic").text
    surface_type = driver.find_element(By.ID, "road_surface").text
    
    # ------ Gathering Parts ------
    
    section_3 = {
        "project_id": project_id,

        "is_new_construction_zone": is_new_construction_zone,

        "highway_district": highway_district,
        "highway_subdistrict": highway_subdistrict,
        "highway_maintenance_unit": highway_maintenance_unit,

        "highway_number": highway_number,
        "control_section": control_section,
        "kilopost": kilopost,

        "road_name": road_name,
        "section_name": section_name,
        "province": province,
        "section_start_km": section_start_km,
        "section_end_km": section_end_km,

        "road_feature": road_feature,
        "road_status": road_status,
        "lanes_count": lanes_count,

        "direction": road_direction,
        "median_type": median_type,
        "traffic_type": traffic_type,
        "surface_type": surface_type,
    }

    section_3_df = pd.DataFrame([section_3])
    
    return section_3_df

## SECTION 4

In [22]:
def section_4(project_id):
    
    road_horizontal = driver.find_element(By.ID, "char_horizontal").text
    road_vertical = driver.find_element(By.ID, "char_vertical").text
    intersection = driver.find_element(By.ID, "char_intersection").text
    median_opening = driver.find_element(By.ID, "char_openacess").text
    connector = driver.find_element(By.ID, "char_connect").text
    special_area = driver.find_element(By.ID, "char_other").text
    
    # ------ Gathering Parts ------
    
    section_4 = {
        "project_id": project_id,
        "road_horizontal": road_horizontal,
        "road_vertical": road_vertical,
        "intersection": intersection,
        "median_opening": median_opening,
        "connector": connector,
        "special_area": special_area,
    }

    section_4_df = pd.DataFrame([section_4])
    
    return section_4_df

## SECTION 5

In [23]:
def get_span_section_5():

    try:
        control_span = wait.until(EC.presence_of_element_located((By.ID, "control")))
        
        driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", control_span)
        time.sleep(1) 
        
        driver.execute_script("arguments[0].click();", control_span)
        print("Clicked on 'control' span successfully.")
        
        time.sleep(2) 
        
    except Exception as e:
        print(f"Cannot click the control span: {e}")

In [24]:
def section_5(project_id):
    
    get_span_section_5()
    
    selected_controls = get_checkbox_labels("control[]")
    
    # ------ Gathering Parts ------
    
    section_5 = {
        "project_id": project_id,
        "speed_sign": 1 if "ป้ายจำกัดความเร็ว" in selected_controls else 0,
        "stop_sign": 1 if "ป้ายบังคับหยุด" in selected_controls else 0,
        "warning_sign": 1 if "ป้ายจราจรประเภทเตือนอื่นๆ" in selected_controls else 0,
        "traffic_light": 1 if "สัญญาณไฟจราจร" in selected_controls else 0,
        "flashing_light": 1 if "สัญญาณไฟกระพริบ" in selected_controls else 0,
        "road_marking": 1 if "เส้นเครื่องหมายจราจรบนผิวทาง" in selected_controls else 0,
        "no_overtake_zone": 1 if "เขตห้ามแซง" in selected_controls else 0,
        "no_parking_zone": 1 if "เขตห้ามจอด" in selected_controls else 0,
        "uncontrolled_crosswalk": 1 if "เป็นทางคนเดินข้ามถนนที่ไม่มีการควบคุม" in selected_controls else 0,
        "controlled_crosswalk": 1 if "เป็นทางคนเดินข้ามถนนที่มีการควบคุม" in selected_controls else 0,
        "pedestrian_bridge": 1 if "สะพานลอยคนเดินข้าม" in selected_controls else 0,
        "optical_speed_bar": 1 if "Optical Speed Bar" in selected_controls else 0,
        "rumble_strip": 1 if "Rumble Strip" in selected_controls else 0,
        "speed_camera": 1 if "กล้องตรวจจับความเร็ว" in selected_controls else 0,
        "guide_sign": 1 if "เครื่องหมายนำทาง" in selected_controls else 0,
        "no_control": 1 if "ไม่มีการควบคุมอย่างใดอย่างหนึ่ง" in selected_controls else 0,
    }

    section_5_df = pd.DataFrame([section_5])
    
    return section_5_df

## SECTION 6-7

In [25]:
def get_time_dropdown(key):
    """ฟังก์ชันเฉพาะสำหรับดึง Text จาก Dropdown (Select)"""
    try:
        try:
            el = driver.find_element(By.ID, key)
        except:
            el = driver.find_element(By.NAME, key)
            
        select = Select(el)
        return select.first_selected_option.text.strip()
    except:
        return ""

In [26]:
def const_section_6_7():
    traffic_map = {
        "1": "น้อยกว่า 500 เมตร",
        "2": "500 เมตร ถึง 1 กิโลเมตร",
        "3": "1 กิโลเมตร ถึง 2 กิโลเมตร",
        "4": "มากกว่า 2 กิโลเมตร",
        "5": "ไม่ติดสะสม"
    }
    
    return traffic_map

In [27]:
def section_6_7(project_id):
    
    traffic_map = const_section_6_7()

    traffic_congestion_no = get_attr_by_id("traffic_condition_parent", "data-value")
    traffic_congestion = traffic_map.get(traffic_congestion_no)

    start_hour = get_time_dropdown("traffic_condition_begin-hour")
    start_min  = get_time_dropdown("traffic_condition_begin-minute")
    end_hour   = get_time_dropdown("traffic_condition_end-hour")
    end_min    = get_time_dropdown("traffic_condition_end-minute")
    start_time = f"{start_hour}:{start_min}"
    end_time = f"{end_hour}:{end_min}"

    road_surface = driver.find_element(By.ID, "env_surface").text
    road_condition = driver.find_element(By.ID, "env_status").text
    weather = driver.find_element(By.ID, "env_weather").text
    lighting = driver.find_element(By.ID, "env_light").text
    
    # ------ Gathering Parts ------
    
    section_6_7 = {
        "project_id": project_id,
        "traffic_congestion": traffic_congestion,
        "start_time": start_time,
        "end_time": end_time,
        
        "road_surface": road_surface,
        "road_condition": road_condition,
        "weather": weather,
        "lighting": lighting,
    }

    section_6_7_df = pd.DataFrame([section_6_7])
    
    return section_6_7_df

## SECTION 8

In [28]:
def const_section_8():
    gender_map = {
        "1": "ชาย",
        "2": "หญิง"
    }

    safety_map = {
        "1": "หมวกนิรภัย",
        "2": "เข็มขัดนิรภัย",
        "3": "ไม่ใช้",
        "4": "ไม่ระบุ"
    }

    injury_map = {
        "1": "ตาย ณ จุดเกิดเหตุ",
        "2": "ตาย ณ โรงพยาบาล",
        "3": "บาดเจ็บสาหัส",
        "4": "บาดเจ็บเล็กน้อย",
        "5": "ไม่ได้รับบาดเจ็บ",
        "6": "ไม่ระบุ"
    }

    drug_map = {
        "1": "มี",
        "2": "ไม่มี",
        "3": "ไม่ระบุ"
    }
    
    return gender_map, safety_map, injury_map, drug_map

In [29]:
def section_8(project_id):
    
    gender_map, safety_map, injury_map, drug_map = const_section_8()
    
    vehicles = []

    vehicle_blocks = soup.select("#vehiclelist li")

    for v in vehicle_blocks:

        def get_value(name):
            el = v.select_one(f'input[name="{name}"]')
            return el["value"].strip() if el else None
        
        # ------ Gathering Parts ------

        item = {
            "project_id": project_id,
            "vehicle_brand": get_value("vehicle_brand[]"),
            "color": get_value("vehicle_color[]"),
            "driver_name": get_value("vehicle_name[]"),
            "driver_citizen_id": get_value("vehicle_id_no[]"),
            "driver_age": get_value("vehicle_age[]"),
            "driver_gender": gender_map.get(get_value("vehicle_sex[]")),
            "safety_equipment": safety_map.get(get_value("vehicle_safety[]")),
            "injury_status": injury_map.get(get_value("vehicle_damage[]")),
        }

        vehicles.append(item)

    section_8_df = pd.DataFrame(vehicles)
    
    return section_8_df

## SECTION 9

In [30]:
def const_section_9():
    categories = {
        "100": "ด้านสภาพแวดล้อม",
        "200": "ด้านยานพาหนะ",
        "300": "ด้านผู้ขับขี่",
        "400": "ด้านคนเดินเท้า"
    }
    
    return categories

In [31]:
def section_9(project_id):
    
    categories = const_section_9()
    
    causes = []

    for code, category_name in categories.items():

        items = soup.select(f"#causelist-{code}-items li")

        for i, item in enumerate(items, start=1):

            cause = item.text.split("\n ")[1].strip()
            
            # ------ Gathering Parts ------

            data = {
                "project_id": project_id,
                "category": category_name,
                "rank": i,
                "cause": cause
            }

            causes.append(data)

    section_9_df = pd.DataFrame(causes)
    
    return section_9_df

## SECTION Diagram

In [32]:
def section_diagram(project_id):
    
    crash_id = driver.find_element(By.ID, "crash").text
    inci_address = soup.select_one("#address").get_text(strip=True).replace("สถานที่:", "")
    gps_loc = soup.select_one("#position").get_text(strip=True).split(', ')
    gps_lat = gps_loc[0]
    gps_lon = gps_loc[1]
    inci_desc = soup.select_one("#summary").get_text(strip=True)
        
    diagram = {
        "project_id": project_id,
        "crash_id": crash_id,
        "incident_address": inci_address,
        "gps_lat": gps_lat,
        "gps_lng": gps_lon,
        "incident_description": inci_desc,
    }

    diagram_df = pd.DataFrame([diagram])
    
    return diagram_df

# Save and Upload

In [33]:
def save_and_upload_data_direct(project_id, drive, main_folder_id, 
                                header_df, section_1_2_df, section_3_df, 
                                section_4_df, section_5_df, section_6_7_df, 
                                section_8_df, section_9_df, diagram_df):

    folder_name = f"{project_id}_dataset"
    print(f"☁️ Creating {folder_name} on Google Drive...")
    
    folder_metadata = {
        'title': folder_name,
        'mimeType': 'application/vnd.google-apps.folder',
        'parents': [{'id': main_folder_id}]
    }
    gdrive_folder = drive.CreateFile(folder_metadata)
    gdrive_folder.Upload()
    new_folder_id = gdrive_folder['id'] 

    files_to_upload = {
        "header.csv": header_df,
        "section_1_2.csv": section_1_2_df,
        "section_3.csv": section_3_df,
        "section_4.csv": section_4_df,
        "section_5.csv": section_5_df,
        "section_6_7.csv": section_6_7_df,
        "section_8.csv": section_8_df,
        "section_9.csv": section_9_df,
        "section_diagram.csv": diagram_df
    }

    print("🚀 Upload data from Memory up to Google Drive directly...")
    
    for filename, df in files_to_upload.items():
        csv_content = df.to_csv(index=False, encoding='utf-8-sig')
        
        file_metadata = {
            'title': filename,
            'parents': [{'id': new_folder_id}]
        }
        
        file_drive = drive.CreateFile(file_metadata)
        
        file_drive.SetContentString(csv_content) 
        file_drive.Upload()

    print(f"🎉 Uploaded data ID: {project_id} to Google Drive successfully!\n")

# Direct Link

In [34]:
# Option 1: ID เรียงต่อเนื่อง
START_ID = 1159341
STOP_ID = 1159345

INCIDENT_IDS = [str(i) for i in range(START_ID, STOP_ID + 1)]

# Option 2: ID ไม่ได้เรียงต่อเนื่อง
# INCIDENT_IDS = ["1158950", "1158955", "1158980"]

## Main

In [35]:
for project_id in INCIDENT_IDS:

    print(f"\n--- Checking Incident ID: {project_id} ---")

    

    target_url = f"https://haims2.doh.go.th/form/{project_id}"

    driver.get(target_url)

    

    try:

        wait.until(EC.presence_of_element_located((By.TAG_NAME, "form")))

        time.sleep(2) 

        

        print(f"✅ Found ID {project_id} let's Extract Data...")

        

        html = driver.page_source

        soup = BeautifulSoup(html, "html.parser")

        header_df=section_0(project_id)
        section_1_2_df=section_1_2(project_id)
        section_3_df=section_3(project_id) 
        section_4_df=section_4(project_id)
        section_5_df=section_5(project_id)
        section_6_7_df=section_6_7(project_id)
        section_8_df=section_8(project_id)
        section_9_df=section_9(project_id)
        diagram_df=section_diagram(project_id)

        save_and_upload_data_direct(
            project_id=project_id, 
            drive=drive, 
            main_folder_id=MAIN_GDRIVE_FOLDER_ID,
            header_df=header_df, 
            section_1_2_df=section_1_2_df, 
            section_3_df=section_3_df, 
            section_4_df=section_4_df, 
            section_5_df=section_5_df, 
            section_6_7_df=section_6_7_df, 
            section_8_df=section_8_df, 
            section_9_df=section_9_df, 
            diagram_df=diagram_df
        )
        

        print(f"💾 Saved ID: {project_id} sucessfully!")

        

    except TimeoutException:

        print(f"⚠️ Skipped ID: {project_id} - Form not found")

        continue 

        

    except Exception as e:

        print(f"❌ Error on ID: {project_id} - {e}")

        continue


--- Checking Incident ID: 1159341 ---
✅ Found ID 1159341 let's Extract Data...
Clicked on 'control' span successfully.
☁️ Creating 1159341_dataset on Google Drive...
🚀 Upload data from Memory up to Google Drive directly...
🎉 Uploaded data ID: 1159341 to Google Drive successfully!

💾 Saved ID: 1159341 sucessfully!

--- Checking Incident ID: 1159342 ---
✅ Found ID 1159342 let's Extract Data...
Clicked on 'control' span successfully.
☁️ Creating 1159342_dataset on Google Drive...
🚀 Upload data from Memory up to Google Drive directly...
🎉 Uploaded data ID: 1159342 to Google Drive successfully!

💾 Saved ID: 1159342 sucessfully!

--- Checking Incident ID: 1159343 ---
✅ Found ID 1159343 let's Extract Data...
Clicked on 'control' span successfully.
☁️ Creating 1159343_dataset on Google Drive...
🚀 Upload data from Memory up to Google Drive directly...
🎉 Uploaded data ID: 1159343 to Google Drive successfully!

💾 Saved ID: 1159343 sucessfully!

--- Checking Incident ID: 1159344 ---
✅ Found ID 11